# 05 Story-disjoint split: seeds 43-46

Trains four more story-disjoint adapters (seeds 43-46) on the fixed story-disjoint split of the main
pipeline, saves them, and evaluates each on the 837-item story-disjoint ToMBench test split and the
four external benchmarks. With the seed-42 run this gives the five-seed story-disjoint results.

- [1] rebuilds the 2,860 ToMBench items with the main pipeline's code and the same
  `GroupShuffleSplit(random_state=42)`, and checks that the items equal the item-level manifests,
  that the split is 1,647 / 376 / 837 with no shared story, and that the test items and their order
  equal the released seed-42 story-disjoint CSVs.
- [2] re-evaluates the seed-42 story-disjoint adapter (`adapter_groupsplit`, the 2026-09-03 run,
  SHA-256 prefix `029b209a`) and compares it with its CSV.
- [3] trains, saves the adapter and the per-epoch log, and evaluates with the model in memory, as
  in the main pipeline. If an adapter exists but some evaluations are missing, it is loaded with
  unsloth and only the missing evaluations are run.
- [4] summary: change by seed with exact McNemar tests (Holm within each seed's five benchmarks),
  and the five-seed mean with a between-seed 95% t-interval.

Run the cells in order: [0-1] installs the pinned package versions; restart the runtime
(Runtime > Restart session) and continue from [0-2]. Results are written to Google Drive, and
cells that find their output already saved skip it, so an interrupted run can be resumed by
running [0-2] again and then the interrupted cell.

`DRIVE_DIR` is the Drive folder used by `notebooks/tombench_slm_qlora_complete_pipeline.ipynb`;
`tier4_v2_fixed/` inside it holds that notebook's outputs (split and manifest files, per-item
CSVs, adapters). The other folders are created by the revision notebooks; see
`notebooks/revision/README.md` for where their files are in `results/`.

## [0] Setup

In [ ]:
# -- [0-1] Install packages (versions pinned to requirements.lock.txt) --
import time, torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Runtime -> Change runtime type -> select a GPU.'); raise SystemExit
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB VRAM)')
t0 = time.time()
get_ipython().system('pip install -q unsloth==2026.8.18 unsloth_zoo==2026.8.12')
get_ipython().system('pip install -q transformers==5.5.0 trl==0.24.0 accelerate==1.14.0 bitsandbytes==0.50.1 peft==0.20.0')
get_ipython().system('pip install -q "datasets==3.6.0" tqdm')
print(f'   {time.time()-t0:.0f}s  ->  restart the runtime, then run from [0-2]')

In [ ]:
# -- [0-2] Mount Drive, set paths, helpers (same functions as the main pipeline notebook) --
from google.colab import drive
drive.mount('/content/drive')

import os, re, glob, math, json as _json, time, hashlib, platform, subprocess
import pandas as pd, numpy as np, torch
import transformers, trl, peft, unsloth
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
transformers.logging.set_verbosity_error()
assert transformers.__version__ == '5.5.0', f'transformers version not pinned: {transformers.__version__}'

DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
OUT_DIR     = f'{DRIVE_DIR}/storydisjoint_5seed'
os.makedirs(OUT_DIR, exist_ok=True)

EXTERNAL = ['ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
NEW_SEEDS = [43, 44, 45, 46]
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
NUM_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2
TEST_RATIO = 0.30
VAL_RATIO_OF_TRAIN = 0.20

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)
ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')

def normalize_text(s): return ' '.join(str(s).split())
def stable_hash(s) -> int: return int(hashlib.sha256(str(s).encode('utf-8')).hexdigest(), 16) & 0xffffffff
def canonical_item_id_base(source, task, story, question):
    key = '|'.join([str(source), str(task), normalize_text(story), normalize_text(question)])
    return hashlib.sha256(key.encode('utf-8')).hexdigest()[:16]
def canonical_story_id(story): return hashlib.sha256(normalize_text(story).encode('utf-8')).hexdigest()[:16]
def prompt_hash_of(up): return hashlib.sha256(str(up).encode('utf-8')).hexdigest()[:16]
class _UniqueIdAssigner:
    def __init__(self): self._counts = {}
    def get(self, source, task, story, question):
        base = canonical_item_id_base(source, task, story, question)
        n = self._counts.get(base, 0); self._counts[base] = n + 1
        return base if n == 0 else f'{base}-{n}'

def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'

def to_text(tokenizer, user_prompt):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_prompt}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def to_sft_text(tokenizer, user_prompt, gold):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_prompt},
            {'role': 'assistant', 'content': f'[[{gold}]]'}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

@torch.no_grad()
def evaluate(eval_records, model, tokenizer, desc='Eval', max_new_tokens=12):
    # same output columns as evaluate() in the main pipeline notebook
    from tqdm.auto import tqdm
    FastLanguageModel.for_inference(model)
    results = []
    for r in tqdm(eval_records, desc=desc):
        prompt = to_text(tokenizer, r['user_prompt'])
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = extract_letter(gen)
        results.append({
            'item_id': r['item_id'], 'prompt_hash': r.get('prompt_hash'),
            'story_id': canonical_story_id(r['story']) if r.get('story') else None,
            'source': r.get('source'), 'task': r.get('task'), 'category': r.get('category'), 'ability': r.get('ability'),
            'answer': r['answer'], 'pred': pred, 'raw': gen, 'correct': int(pred == r['answer']),
            'n_options': r.get('n_options'), 'marker_found': '[[' in gen,
        })
    return pd.DataFrame(results)

def get_batch_sizes():
    _gpu = torch.cuda.get_device_name(0)
    _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if 'A100' in _gpu or _vram >= 35: return 8, 1, 16
    elif 'L4' in _gpu or _vram >= 22: return 4, 2, 8
    else: return 2, 4, 4

def train_qlora(train_records, val_records, seed, output_dir):
    # same hyperparameters as train_and_eval_seed() in the main pipeline notebook
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
                                                         dtype=None, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(
        model, r=16, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0.0, bias='none', use_gradient_checkpointing='unsloth', random_state=seed)
    train_bs, grad_accum, eval_bs = get_batch_sizes()
    train_ds = Dataset.from_dict({'text': [to_sft_text(tokenizer, r['user_prompt'], r['answer']) for r in train_records]})
    val_ds   = Dataset.from_dict({'text': [to_sft_text(tokenizer, r['user_prompt'], r['answer']) for r in val_records]})
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds,
        dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2, packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=train_bs, per_device_eval_batch_size=eval_bs,
            gradient_accumulation_steps=grad_accum, warmup_ratio=0.03, num_train_epochs=NUM_EPOCHS,
            learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
            logging_steps=20, eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
            metric_for_best_model='eval_loss', greater_is_better=False, save_total_limit=2,
            optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine', seed=seed,
            output_dir=output_dir, report_to='none'),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    t0 = time.time()
    trainer.train()
    # keep the per-epoch validation loss
    log = pd.DataFrame(trainer.state.log_history)
    best = trainer.state.best_model_checkpoint
    print(f'seed {seed} training done ({(time.time()-t0)/60:.1f} min), best checkpoint: {best}')
    del trainer
    return model, tokenizer, log, best

def load_external(bm):
    with open(f'{RESULTS_DIR}/manifest_{bm}.json', encoding='utf-8') as f:
        return _json.load(f)
EXT_RECORDS = {bm: load_external(bm) for bm in EXTERNAL}
print({bm: len(v) for bm, v in EXT_RECORDS.items()})

meta = {'gpu': torch.cuda.get_device_name(0), 'batch_sizes(train,accum,eval)': get_batch_sizes(),
        'bf16': bool(is_bfloat16_supported()), 'cuda': torch.version.cuda, 'torch': torch.__version__,
        'transformers': transformers.__version__, 'trl': trl.__version__, 'peft': peft.__version__,
        'unsloth': unsloth.__version__, 'python': platform.python_version(), 'started': time.strftime('%Y-%m-%d %H:%M:%S')}
with open(f'{OUT_DIR}/session_meta_{time.strftime("%Y%m%d_%H%M")}.json', 'w') as f:
    _json.dump(meta, f, indent=2)
print(meta)

## [1] Rebuild and check the story-disjoint split

In [ ]:
# -- [1] Read the ToMBench source files (same path search as the main pipeline; clone from GitHub if absent) --
from sklearn.model_selection import GroupShuffleSplit

TOMBENCH_DATA_PATHS = ['/content/ToMBench/data', '/content/data',
                       '/content/drive/MyDrive/ToMBench-main/data', '/content/drive/MyDrive/ToMBench/data']
TOMBENCH_DIR = next((p for p in TOMBENCH_DATA_PATHS if os.path.isdir(p) and glob.glob(os.path.join(p, '*.jsonl'))), None)
if TOMBENCH_DIR is None:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/zhchen18/ToMBench.git', '/content/ToMBench'], check=False)
    TOMBENCH_DIR = '/content/ToMBench/data'
print('ToMBench data dir:', TOMBENCH_DIR)

def is_valid(x):
    if x is None: return False
    if isinstance(x, float) and math.isnan(x): return False
    s = str(x).strip(); return len(s) > 0 and s.lower() != 'nan'
def normalize_ability(ab): return ab.replace('Non-literal', 'Non-Literal')
def build_user_prompt(story, question, opts):
    return f'[Story]\n{story}\n\n[Question]\n{question}\n\n[Candidate Answers]\n' + '\n'.join(f'{l}. {t}' for l, t in opts)

_ids = _UniqueIdAssigner(); records = []
for fp in sorted(glob.glob(os.path.join(TOMBENCH_DIR, '*.jsonl'))):
    task = os.path.splitext(os.path.basename(fp))[0]
    with open(fp, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = _json.loads(line)
            ability = normalize_ability(row.get('能力\nABILITY', 'Unknown'))
            ans = row.get('答案\nANSWER') or row.get('ANSWER')
            if not is_valid(row.get('STORY')) or not is_valid(row.get('QUESTION')) or not is_valid(ans): continue
            opts = [(L, str(row.get(f'OPTION-{L}')).strip()) for L in 'ABCD' if is_valid(row.get(f'OPTION-{L}'))]
            up = build_user_prompt(row['STORY'], row['QUESTION'], opts)
            records.append({'source': 'ToMBench', 'task': task, 'category': ability.split(':')[0].strip(), 'ability': ability,
                            'story': row['STORY'], 'question': row['QUESTION'], 'user_prompt': up,
                            'answer': str(ans).strip().upper()[0],
                            'item_id': _ids.get('ToMBench', task, row['STORY'], row['QUESTION']),
                            'prompt_hash': prompt_hash_of(up), 'n_options': len(opts)})
df = pd.DataFrame(records)
assert len(df) == 2860 and not df['item_id'].duplicated().any()

# check 1: same items as the item-level manifests
man = []
for f in ['split_train_full_v2.json', 'split_val_full_v2.json', 'split_test_full_v2.json']:
    with open(f'{RESULTS_DIR}/{f}', encoding='utf-8') as fh: man += _json.load(fh)
man = pd.DataFrame(man)[['item_id', 'prompt_hash', 'answer']]
chk = df.merge(man, on='item_id', suffixes=('', '_m'))
assert len(chk) == 2860 and (chk['prompt_hash'] == chk['prompt_hash_m']).all() and (chk['answer'] == chk['answer_m']).all()
print('check 1 passed: 2,860 items, item_id, prompt_hash and answer identical')

# story-disjoint split (same code as the group-split cell of the main pipeline notebook)
df_g = df.copy()
df_g['canonical_story_id'] = df_g['story'].apply(canonical_story_id)
gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=42)
trainval_idx, test_idx = next(gss1.split(df_g, groups=df_g['canonical_story_id'].values))
trainval_df_g = df_g.iloc[trainval_idx].reset_index(drop=True)
gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_RATIO_OF_TRAIN, random_state=42)
train_idx2, val_idx2 = next(gss2.split(trainval_df_g, groups=trainval_df_g['canonical_story_id'].values))
train_df_g = trainval_df_g.iloc[train_idx2].reset_index(drop=True)
val_df_g   = trainval_df_g.iloc[val_idx2].reset_index(drop=True)
test_df_g  = df_g.iloc[test_idx].reset_index(drop=True)

# check 2: sizes and story overlap
assert (len(train_df_g), len(val_df_g), len(test_df_g)) == (1647, 376, 837), (len(train_df_g), len(val_df_g), len(test_df_g))
for a, b in [(train_df_g, val_df_g), (train_df_g, test_df_g), (val_df_g, test_df_g)]:
    assert not (set(a['canonical_story_id']) & set(b['canonical_story_id']))
print('check 2 passed: 1,647 / 376 / 837, no story overlap')

# check 3: same test items, in the same order, as the released seed-42 story-disjoint CSVs
pub_base = pd.read_csv(f'{RESULTS_DIR}/groupsplit_base_ToMBench.csv')
pub_tuned = pd.read_csv(f'{RESULTS_DIR}/groupsplit_finetuned_ToMBench.csv')
assert list(pub_base['item_id']) == list(test_df_g['item_id'])
assert set(pub_tuned['item_id']) == set(test_df_g['item_id'])
print('check 3 passed: the 837 test items match the released seed-42 story-disjoint CSVs, in the same order')
print(f'seed-42 story-disjoint ToMBench accuracy in the results folder: {pub_tuned["correct"].mean()*100:.2f}%')

STORY_TRAIN = train_df_g.to_dict('records')
STORY_VAL   = val_df_g.to_dict('records')
STORY_TEST  = test_df_g.to_dict('records')

## [2] Seed-42 story-disjoint adapter

In [ ]:
# -- [2] Check that the seed-42 adapter reproduces its released predictions --
ADAPTER_GS42 = f'{RESULTS_DIR}/adapter_groupsplit'
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()[:16]
print('adapter_groupsplit sha256:', sha16(f'{ADAPTER_GS42}/adapter_model.safetensors'), '(029b209a2f1cfc5b)')

p42 = f'{OUT_DIR}/check_seed42_groupsplit_ToMBench.csv'
if not os.path.exists(p42):
    m42, t42 = FastLanguageModel.from_pretrained(model_name=ADAPTER_GS42, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    evaluate(STORY_TEST, m42, t42, desc='seed42-groupsplit-check').to_csv(p42, index=False)
    del m42, t42; torch.cuda.empty_cache()
chk42 = pd.read_csv(p42).merge(pub_tuned[['item_id', 'pred']], on='item_id', suffixes=('', '_pub'))
n_mis = int((chk42['pred'] != chk42['pred_pub']).sum())
print(f'seed42 story-disjoint re-evaluation vs results-folder CSV: mismatches {n_mis}/{len(chk42)}, accuracy {chk42["correct"].mean()*100:.2f}%')
if n_mis:
    print('Non-zero -- check this before running [3].')

## [3] Seeds 43-46: training and evaluation

Outputs: `groupsplit_finetuned_seed<N>_<benchmark>.csv` (same columns as the seed-42 files),
`adapter_groupsplit_seed<N>/`, `trainlog_groupsplit_seed<N>.csv`.

In [ ]:
# -- [3] seed43~46 --
def res_path(s, bm): return f'{OUT_DIR}/groupsplit_finetuned_seed{s}_{bm}.csv'
EVAL_SETS = [('ToMBench', STORY_TEST)] + [(bm, EXT_RECORDS[bm]) for bm in EXTERNAL]

for s in NEW_SEEDS:
    adapter_dir = f'{OUT_DIR}/adapter_groupsplit_seed{s}'
    todo = [(bm, recs) for bm, recs in EVAL_SETS if not os.path.exists(res_path(s, bm))]
    if not todo:
        print(f'[skip] seed{s}: all done'); continue
    if os.path.exists(f'{adapter_dir}/adapter_model.safetensors'):
        print(f'seed{s}: adapter present -> loading it with unsloth to finish the remaining evaluations')
        model, tok = FastLanguageModel.from_pretrained(model_name=adapter_dir, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    else:
        print('=' * 60); print(f'story-disjoint seed {s}: training'); print('=' * 60)
        model, tok, log, best = train_qlora(STORY_TRAIN, STORY_VAL, s, f'outputs_groupsplit_seed{s}')
        model.save_pretrained(adapter_dir); tok.save_pretrained(adapter_dir)
        log.to_csv(f'{OUT_DIR}/trainlog_groupsplit_seed{s}.csv', index=False)
        with open(f'{OUT_DIR}/trainlog_groupsplit_seed{s}_best.txt', 'w') as f: f.write(str(best))
        print(f'adapter saved: {adapter_dir}')
    for bm, recs in todo:
        df_r = evaluate(recs, model, tok, desc=f'seed{s}-{bm}')
        df_r['condition'] = 'groupsplit_finetuned'; df_r['seed'] = s
        df_r.to_csv(res_path(s, bm), index=False)
        print(f'seed{s} {bm}: {df_r["correct"].mean()*100:.2f}%')
    del model, tok; torch.cuda.empty_cache()
print('\n[3] done')

## [4] Summary (no GPU needed)

Base: `groupsplit_base_ToMBench.csv` for ToMBench, `baseline_*.csv` for the external benchmarks.
The item-level five-seed change is shown alongside.

In [ ]:
# -- [4] Summary --
from google.colab import drive
drive.mount('/content/drive')
import os, pandas as pd, numpy as np
from scipy.stats import binomtest
pd.set_option('display.width', 200)
DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
OUT_DIR     = f'{DRIVE_DIR}/storydisjoint_5seed'
BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
SEEDS = [42, 43, 44, 45, 46]

def base_path(bm):
    return f'{RESULTS_DIR}/groupsplit_base_ToMBench.csv' if bm == 'ToMBench' else f'{RESULTS_DIR}/baseline_{bm}.csv'
def tuned_path(s, bm):
    return f'{RESULTS_DIR}/groupsplit_finetuned_{bm}.csv' if s == 42 else f'{OUT_DIR}/groupsplit_finetuned_seed{s}_{bm}.csv'

def holm(ps):
    order = np.argsort(ps); adj = np.empty(len(ps)); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (len(ps) - rank) * ps[i])); adj[i] = run
    return adj

rows = []
for s in SEEDS:
    seed_rows = []
    for bm in BENCHMARKS:
        if not os.path.exists(tuned_path(s, bm)):
            print(f'[missing] seed{s} {bm}'); continue
        b = pd.read_csv(base_path(bm)); t = pd.read_csv(tuned_path(s, bm))
        m = b.merge(t, on='item_id', suffixes=('_b', '_t'), validate='one_to_one')
        assert len(m) == len(b) == len(t), f'{s} {bm}: item sets differ'
        n01 = int(((m['correct_b'] == 0) & (m['correct_t'] == 1)).sum())
        n10 = int(((m['correct_b'] == 1) & (m['correct_t'] == 0)).sum())
        p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
        seed_rows.append({'seed': s, 'benchmark': bm, 'n': len(m), 'base_acc': m['correct_b'].mean() * 100,
                          'tuned_acc': m['correct_t'].mean() * 100,
                          'delta_pp': (m['correct_t'].mean() - m['correct_b'].mean()) * 100,
                          'improved': n01, 'worsened': n10, 'mcnemar_p': p})
    if seed_rows:
        adj = holm(np.array([r['mcnemar_p'] for r in seed_rows]))
        for r, a in zip(seed_rows, adj): r['holm_p'] = a; r['sig_holm'] = a < 0.05
    rows += seed_rows
res = pd.DataFrame(rows)
res.to_csv(f'{OUT_DIR}/storydisjoint_5seed_by_seed.csv', index=False)
print('[story-disjoint change by seed, exact McNemar + Holm]')
print(res.round(3).to_string(index=False))

agg = res.groupby('benchmark').agg(delta_mean=('delta_pp', 'mean'), delta_sd=('delta_pp', 'std'),
                                   n_seeds=('seed', 'count'), sig_seeds=('sig_holm', 'sum'),
                                   positive_seeds=('delta_pp', lambda x: int((x > 0).sum()))).reindex(BENCHMARKS)
# a direction is stated only when the 95% t-interval of the five-seed mean excludes 0
from scipy.stats import t as _t
_half = _t.ppf(0.975, agg['n_seeds'] - 1) * agg['delta_sd'] / np.sqrt(agg['n_seeds'])
agg['ci95_lo'] = agg['delta_mean'] - _half; agg['ci95_hi'] = agg['delta_mean'] + _half
agg['ci_excludes_0'] = (agg['ci95_lo'] > 0) | (agg['ci95_hi'] < 0)

# comparison: item-level five-seed change (released CSVs)
il = []
for bm in BENCHMARKS:
    b = pd.read_csv(f'{RESULTS_DIR}/baseline_{bm}.csv')
    for s in SEEDS:
        t = pd.read_csv(f'{RESULTS_DIR}/finetuned_seed{s}_{bm}.csv')
        il.append({'benchmark': bm, 'seed': s, 'delta': (t['correct'].mean() - b['correct'].mean()) * 100})
il = pd.DataFrame(il).groupby('benchmark')['delta'].agg(['mean', 'std']).reindex(BENCHMARKS)
agg['itemlevel_mean'] = il['mean']; agg['itemlevel_sd'] = il['std']
print('\nseed42 = the 2026-09-03 run whose adapter was retained')
print('\n[five-seed summary] story-disjoint vs item-level (pp)')
print(agg.round(2).to_string())
agg.to_csv(f'{OUT_DIR}/storydisjoint_5seed_summary.csv')